# Imports

In [70]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE, RandomOverSampler

# Loading data and exploring it 

In [71]:
train_df =pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-training-master.csv")
test_df = pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-testing-master.csv")

In [72]:
# Training data
print("First 5 rows of the training data:")
train_df.head()

First 5 rows of the training data:


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


In [73]:
train_df.describe()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
count,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000
mean,225398.667955,39.373153,31.256336,15.807494,3.604437,12.965722,631.616223,14.480868,0.567107
std,129531.918550,12.442369,17.255727,8.586242,3.070218,8.258063,240.803001,8.596208,0.495477
min,2.000000,18.000000,1.000000,1.000000,0.000000,0.000000,100.000000,1.000000,0.000000
25%,113621.750000,29.000000,16.000000,9.000000,1.000000,6.000000,480.000000,7.000000,0.000000
50%,226125.500000,39.000000,32.000000,16.000000,3.000000,12.000000,661.000000,14.000000,1.000000
75%,337739.250000,48.000000,46.000000,23.000000,6.000000,19.000000,830.000000,22.000000,1.000000
max,449999.000000,65.000000,60.000000,30.000000,10.000000,30.000000,1000.000000,30.000000,1.000000


In [74]:
train_df['Churn'].value_counts()
train_df.isnull().sum()
train_df.info



<bound method DataFrame.info of         CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  \
0              2.0  30.0  Female    39.0             14.0            5.0   
1              3.0  65.0  Female    49.0              1.0           10.0   
2              4.0  55.0  Female    14.0              4.0            6.0   
3              5.0  58.0    Male    38.0             21.0            7.0   
4              6.0  23.0    Male    32.0             20.0            5.0   
...            ...   ...     ...     ...              ...            ...   
440828    449995.0  42.0    Male    54.0             15.0            1.0   
440829    449996.0  25.0  Female     8.0             13.0            1.0   
440830    449997.0  26.0    Male    35.0             27.0            1.0   
440831    449998.0  28.0    Male    55.0             14.0            2.0   
440832    449999.0  31.0    Male    48.0             20.0            1.0   

        Payment Delay Subscription Type Contract Length

In [75]:
train_df.isnull().sum()
train_df.dtypes

CustomerID           float64
Age                  float64
Gender                object
Tenure               float64
Usage Frequency      float64
Support Calls        float64
Payment Delay        float64
Subscription Type     object
Contract Length       object
Total Spend          float64
Last Interaction     float64
Churn                float64
dtype: object

In [76]:
train_df['Churn'] = train_df['Churn'].astype('object')
test_df['Churn'] = test_df['Churn'].astype('object')

In [77]:
# Check unique values in 'Churn' before any changes
print("Unique values in 'Churn' before processing:")
print(train_df['Churn'].unique())
print("Value counts:")
print(train_df['Churn'].value_counts())

Unique values in 'Churn' before processing:
[1.0 0.0 nan]
Value counts:
Churn
1.0    249999
0.0    190833
Name: count, dtype: int64


# Data and feature Engineering 

In [78]:
# Handling missing values for numerical columns by filling them with the mean of each column
train_df.fillna(train_df.median(numeric_only= True), inplace=True)
test_df.fillna(train_df.median(numeric_only= True), inplace=True)

#filling missing values for categorical columns with "Na"
train_df.fillna("Na", inplace=True)
test_df.fillna("Na", inplace=True)


In [80]:
#Encoding categorical variables using label encoder

for col in train_df.select_dtypes(include=['object']).columns:
    # Ensure the column is string type to avoid mixed types
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)
    
    # Combine unique values from train and test to fit encoder
    all_values = pd.concat([train_df[col], test_df[col]]).unique()
    
    # Create a new encoder for each column and fit on all values
    le = LabelEncoder()
    le.fit(all_values)
    
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

In [81]:
#Splitting features and target variable
x = train_df.drop('Churn', axis=1)
y = train_df['Churn']

# Check class distribution
print("Class distribution in y:")
print(y.value_counts())

#Handling class imbalance using RandomOverSampler (works even with 1 sample per class)
ros = RandomOverSampler(random_state=42)
x_res, y_res = ros.fit_resample(x, y)

Class distribution in y:
Churn
1    249999
0    190833
2         1
Name: count, dtype: int64
